In [ ]:
from gen_data_and_simulate_drift import DriftIntensity, DriftSimulator, GenerateFakeData
from training_script import Classifier

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
DATA_PATH = '../datasets/'
MODEL_PATH = '../models/'

### Exploring the generator class

In [ ]:
pokemon_test_data='bank_num.csv'
df = pd.read_csv(DATA_PATH + pokemon_test_data )

In [ ]:
df.info()

In [ ]:
SAMPLE_SIZE=1000
target = 'default'
genertor_fake_data = GenerateFakeData(path_ref_data=DATA_PATH+'bank_num.csv', sample_size=SAMPLE_SIZE, target=target, model_name = 'bank_generator.pkl')

In [ ]:
genertor_fake_data.df_ref.info()

In [ ]:
genertor_fake_data.df_samples.info()

In [ ]:
genertor_fake_data.df_samples.head()

In [ ]:
sampled_data = genertor_fake_data.get_dataclass_sampling()

In [ ]:
sampled_data.train_data.describe()

In [ ]:
sampled_data.test_data.describe()

In [ ]:
sampled_data.list_num_col

In [ ]:
pok_classifier = Classifier(df_train=sampled_data.train_data,
                 num_features=sampled_data.list_num_col,
                 cat_features=None,
                 target=target,
                pkl_file_path=MODEL_PATH + f'class_{target}_model.pkl',
                iterations = 200, depth = None, learning_rate = 0.03)

In [ ]:
pok_classifier.train()

In [ ]:
pok_classifier.serialize()

In [ ]:
prediction = pok_classifier.predict(sampled_data.train_data.drop(target, axis = 1))
#prediction = [1 if e=='True' else 0 for e in prediction]
prediction[:4]

In [ ]:
(prediction==sampled_data.train_data[target].values).mean()

In [ ]:
prediction = pok_classifier.predict(sampled_data.test_data.drop(target, axis = 1))
#prediction = [1 if e=='True' else 0 for e in prediction]
(prediction==sampled_data.test_data[target].values).mean()

### Analyzing the drift

In [ ]:
from gen_data_and_simulate_drift import get_shift_coef

In [ ]:
drift_sim_info = DriftSimulator(sampled_data, nb_cols_to_drift=1)

In [ ]:
df_test_drifted = drift_sim_info.get_test_data_drifted()

In [ ]:
df_test_drifted.job.describe()

In [ ]:
prediction = pok_classifier.predict(df_test_drifted.drop(target, axis = 1))
#prediction = [1 if e=='True' else 0 for e in prediction]
(prediction==sampled_data.test_data[target].values).mean()

In [ ]:
sampled_data.test_data.job.describe()

### Creating marketing dataset

In [ ]:
import os

In [ ]:
os.path.exists(DATA_PATH + 'bank.csv')

In [ ]:
bank = pd.read_csv(DATA_PATH + 'bank.csv')

In [ ]:
bank.info()

In [ ]:
cat_cols = bank.select_dtypes('O').columns
cat_cols = cat_cols.drop('default')
cat_cols

In [ ]:
oe = OrdinalEncoder(dtype=np.int64, handle_unknown='use_encoded_value', unknown_value=-1)

In [ ]:
bank[cat_cols].head()

In [ ]:
bank[cat_cols] = oe.fit_transform(bank[cat_cols])

In [ ]:
bank[cat_cols].head()

In [ ]:
bank.head()

In [ ]:
joblib.dump(oe, MODEL_PATH+ 'bank_ordinal_encoder.pkl')

In [ ]:
bank.to_csv(DATA_PATH + 'bank_num.csv', index = False)

In [ ]:
bank.dtypes